In [59]:
import pandas as pd
import os
import datetime
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pickle 

In [4]:
HWR_PATH = "RanSMAP/dataset/original/i3-gen12/ddr4-2133-16g" #"RanSMAP/dataset/original/i3-gen12/ddr4-3200-16g"

DATASET = "hiranomanabu/ransmap-2024-ransomware-behavioral-features"

strg_access_column_names = ['unix_time_s', 'unix_time_ns', 'lba', 'storage_size', 'storage_entropy']
mem_access_columns_names = ['unix_time_s', 'unix_time_ns', 'gpa', 'mem_size', 'mem_entropy', 'mem_access_type']

ransomware = set(["WannaCry", "Ryuk", "REvil", "LockBit", "Darkside", "Conti"])

ata_col_ind_to_name_map = dict(zip(range(len(strg_access_column_names)), strg_access_column_names))
mem_col_ind_to_name_map = dict(zip(range(len(mem_access_columns_names)), mem_access_columns_names))

kaggle_files_to_load = {
    'ata_read': ata_col_ind_to_name_map,
    'ata_write': ata_col_ind_to_name_map,
    'mem_read': mem_col_ind_to_name_map,
    'mem_write': mem_col_ind_to_name_map,
    'mem_readwrite': mem_col_ind_to_name_map,
    'mem_exec': mem_col_ind_to_name_map
}

NUMBER_OF_OPERATION_LOGS_TO_LOAD = 1

kagglehub.login()

## Get Operations_logs_to_load

In [5]:
parent_folder_path = 'RanSMAP\\dataset\\original\\i3-gen12\\ddr4-2133-16g'
operation_names = os.listdir(parent_folder_path)
operation_paths = [os.path.join(parent_folder_path, operation) for operation in operation_names]

In [6]:
operations_logs_path = {}

for operation, operation_path in zip(operation_names, operation_paths):
    logs_path = [f"{operation}/{log_name}" for log_name in os.listdir(operation_path)]
    operations_logs_path[operation] = logs_path    

In [7]:
operations_logs_to_load = {}

if NUMBER_OF_OPERATION_LOGS_TO_LOAD == -1:
    operations_logs_to_load = operations_logs_path
else:
    for operation, logs in operations_logs_path.items():
        operations_logs_to_load[operation] = logs[:min(len(logs), NUMBER_OF_OPERATION_LOGS_TO_LOAD)]

In [8]:
operations_logs_to_load

{'AESCrypt': ['AESCrypt/AESCrypt-20230524_20-18-05'],
 'Conti': ['Conti/Conti-20230426_23-09-03'],
 'Darkside': ['Darkside/Darkside-20230512_00-06-29'],
 'Firefox': ['Firefox/Firefox-20230428_00-35-50'],
 'Idle': ['Idle/Idle-20230518_22-56-17'],
 'LockBit': ['LockBit/LockBit-20230517_21-35-46'],
 'Office': ['Office/Office-20230525_23-11-40'],
 'REvil': ['REvil/REvil-20230427_23-00-37'],
 'Ryuk': ['Ryuk/Ryuk-20230510_23-53-48'],
 'SDelete': ['SDelete/SDelete-20230519_00-20-40'],
 'WannaCry': ['WannaCry/WannaCry-20230510_21-11-33'],
 'Zip': ['Zip/Zip-20230524_22-57-19']}

## Read Dataset

In [23]:
folder_path = "RanSMAP\\dataset\\original\\i3-gen12\\ddr4-2133-16g\\AESCrypt\\AESCrypt-20230524_20-18-05"

In [98]:
strg_read = pd.read_csv(os.path.join(folder_path, 'ata_read.csv'), names = strg_access_column_names, usecols=[i for i in range(5)])
strg_write = pd.read_csv(os.path.join(folder_path, 'ata_write.csv'), names = strg_access_column_names, usecols=[i for i in range(5)])
mem_read = pd.read_csv(os.path.join(folder_path, 'mem_read.csv'), names = mem_access_columns_names)
mem_write = pd.read_csv(os.path.join(folder_path, 'mem_write.csv'), names = mem_access_columns_names)
mem_read_write = pd.read_csv(os.path.join(folder_path, 'mem_readwrite.csv'), names = mem_access_columns_names)
mem_exec = pd.read_csv(os.path.join(folder_path, 'mem_exec.csv'), names = mem_access_columns_names)

In [25]:
strg_read.head()

,unix_time_s,unix_time_ns,lba,storage_size,storage_entropy
0,1684893978,693442531,38952048,4096,-1
1,1684893978,693442531,38952056,4096,-1
2,1684893978,693443604,38952064,4096,-1
3,1684893978,693443604,38952072,4096,-1
4,1684893978,693444678,38952080,4096,-1


In [ ]:
strg_read_mod = pd.read_csv('RanSMAP\\dataset\\original\\i3-gen12\\ddr4-2133-16g\\AESCrypt\\AESCrypt-20230524_20-18-05\\modified\\ata_read.csv')

In [39]:
strg_read_mod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 447312 entries, 0 to 447311
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   unix_time_s      447312 non-null  int64 
 1   unix_time_ns     447312 non-null  int64 
 2   lba              447312 non-null  int64 
 3   storage_size     447312 non-null  int64 
 4   storage_entropy  447312 non-null  int64 
 5   action           447312 non-null  object
 6   timestamp        447312 non-null  object
 7   relative_time    447312 non-null  object
dtypes: int64(5), object(3)
memory usage: 27.3+ MB


In [99]:
strg_read['action'] = 'ata_read'
strg_write['action'] = 'ata_write'
mem_read['action'] = 'mem_read'
mem_write['action'] = 'mem_write'
mem_read_write['action'] = 'mem_read_write'
mem_exec['action'] = 'mem_exec'

In [301]:
min(tmp_comb['timestamp']), max(tmp_comb['timestamp']), max(tmp_comb['timestamp']) - min(tmp_comb['timestamp'])

(Timestamp('2023-05-23 22:06:17.194543354'),
 Timestamp('2023-05-23 22:09:37.994055700'),
 Timedelta('0 days 00:03:20.799512346'))

In [283]:
mem_read.head()

,unix_time_s,unix_time_ns,gpa,mem_size,mem_entropy,mem_access_type,action,timestamp,relative_time
0,1684893977,194543354,7879543992,0,-1,2,mem_read,2023-05-23 22:06:17.194543354,0 days 00:00:00
1,1684893977,194554089,66838552,0,-1,2,mem_read,2023-05-23 22:06:17.194554089,0 days 00:00:00.000010735
2,1684893977,194564824,71733016,0,-1,2,mem_read,2023-05-23 22:06:17.194564824,0 days 00:00:00.000021470
3,1684893977,194574485,70979088,0,-1,2,mem_read,2023-05-23 22:06:17.194574485,0 days 00:00:00.000031131
4,1684893977,194585220,19186048,0,-1,2,mem_read,2023-05-23 22:06:17.194585220,0 days 00:00:00.000041866


In [143]:
mem_write.head()

,unix_time_s,unix_time_ns,gpa,mem_size,mem_entropy,mem_access_type,action
0,1684893977,194817096,18618559496,4096,0.819862,2,mem_write
1,1684893977,198389704,4276093104,4096,0.170819,2,mem_write
2,1684893977,198910352,6470762232,4096,0.523777,2,mem_write
3,1684893977,199257092,6474102816,4096,0.701360,2,mem_write
4,1684893977,200457265,19204565860,4096,0.835809,2,mem_write


In [80]:
def get_combined_timestamp(row):
    return pd.to_datetime(datetime.datetime.fromtimestamp(row['unix_time_s']).strftime("%Y-%m-%dT%H:%M:%S") + "." + str(row['unix_time_ns']))

In [ ]:
# def time_difference(row, start_point):
#     '''
#     Time Difference between current element (row) and some previous point (start_point)
#     '''
#     row_s = row['unix_time_s']
#     row_ns = row['unix_time_ns']
#     if row['unix_time_ns'] < start_point['unix_time_ns']:
#         row_s -= 1
#         row_ns = 1e9 + row['unix_time_ns']
        
#     ns_diff = row_ns - start_point['unix_time_ns']
#     s_diff = row_s - start_point['unix_time_s']
#     print(s_diff, ": ", ns_diff)
#     return pd.to_datetime(datetime.datetime.fromtimestamp(s_diff).strftime("%Y-%m-%dT%H:%M:%S") + "." + str(ns_diff))

    
#     # return pd.to_datetime(s_diff + ns_diff)

In [45]:
def get_datetime_min(time1, time2):
    time1_s, time1_ns = time1.unix_time_s, time1.unix_time_ns
    time2_s, time2_ns = time2.unix_time_s, time2.unix_time_ns
    
    if time1_s < time2_s:
        return time1
    elif time2_s < time1_s:
        return time2
    else:
        if time1_ns <= time2_ns:
            return time1
        return time2

In [89]:
def compute_relative_time(df, start_sec, start_nsec):
    """
    Compute relative time (in seconds and nanoseconds) from a start point.
    Keeps full nanosecond precision.
    """
    # Convert everything to total nanoseconds from the start point
    total_ns = (df['unix_time_s'] - start_sec) * 1_000_000_000 + (df['unix_time_ns'] - start_nsec)

    df['relative_time'] = pd.to_timedelta(total_ns, unit='ns')
    return df

In [90]:
start_point = get_datetime_min(
    get_datetime_min(
        get_datetime_min(
            get_datetime_min(
                get_datetime_min(strg_read.loc[0, ['unix_time_s', 'unix_time_ns']], strg_write.loc[0, ['unix_time_s', 'unix_time_ns']]), 
                mem_read.loc[0, ['unix_time_s', 'unix_time_ns']]), 
            mem_write.loc[0, ['unix_time_s', 'unix_time_ns']]), 
        mem_read_write.loc[0, ['unix_time_s', 'unix_time_ns']]), 
    mem_exec.loc[0, ['unix_time_s', 'unix_time_ns']])
start_point

unix_time_s     1684893977
unix_time_ns     194543354
Name: 0, dtype: object

In [91]:
start_point.unix_time_s, start_point.unix_time_ns

(np.int64(1684893977), np.int64(194543354))

In [100]:
# start_sec, start_nsec = start_point.unix_time_s
strg_read_new = compute_relative_time(strg_read, start_point.unix_time_s, start_point.unix_time_ns)
strg_write_new = compute_relative_time(strg_write, start_point.unix_time_s, start_point.unix_time_ns)
mem_read_new = compute_relative_time(mem_read, start_point.unix_time_s, start_point.unix_time_ns)
mem_write_new = compute_relative_time(mem_write, start_point.unix_time_s, start_point.unix_time_ns)
mem_read_write_new = compute_relative_time(mem_read_write, start_point.unix_time_s, start_point.unix_time_ns)
mem_exec_new = compute_relative_time(mem_exec, start_point.unix_time_s, start_point.unix_time_ns)

In [93]:
start_point_datetime = get_combined_timestamp(start_point)
start_point_datetime

Timestamp('2023-05-23 22:06:17.194543354')

In [148]:
strg_read.head()

,unix_time_s,unix_time_ns,lba,storage_size,storage_entropy,action,timestamp
0,1684893978,693442531,38952048,4096,-1,ata_read,2023-05-23 22:06:18.693442531
1,1684893978,693442531,38952056,4096,-1,ata_read,2023-05-23 22:06:18.693442531
2,1684893978,693443604,38952064,4096,-1,ata_read,2023-05-23 22:06:18.693443604
3,1684893978,693443604,38952072,4096,-1,ata_read,2023-05-23 22:06:18.693443604
4,1684893978,693444678,38952080,4096,-1,ata_read,2023-05-23 22:06:18.693444678


In [ ]:
strg_read['timestamp'] = strg_read[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
strg_write['timestamp'] = strg_write[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
mem_read['timestamp'] = mem_read[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
mem_write['timestamp'] = mem_write[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
mem_read_write['timestamp'] = mem_read_write[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
mem_exec['timestamp'] = mem_exec[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)

KeyboardInterrupt: 

In [282]:
strg_read.loc[len(strg_read)-1, 'timestamp']

Timestamp('2023-05-23 22:09:37.453444312')

In [ ]:
strg_read['relative_time'] = strg_read['timestamp'].apply(lambda x: x - start_point_datetime)
strg_write['relative_time'] = strg_write['timestamp'].apply(lambda x: x - start_point_datetime)
mem_read['relative_time'] = mem_read[['timestamp']].apply(lambda x: x - start_point_datetime)
mem_write['relative_time'] = mem_write[['timestamp']].apply(lambda x: x - start_point_datetime)
mem_read_write['relative_time'] = mem_read_write[['timestamp']].apply(lambda x: x - start_point_datetime)
mem_exec['relative_time'] = mem_exec[['timestamp']].apply(lambda x: x - start_point_datetime)

In [49]:
start_point.unix_time_ns

np.int64(194543354)

In [58]:
mem_read.head()

,unix_time_s,unix_time_ns,gpa,mem_size,mem_entropy,mem_access_type,relative_time
0,1684893977,194543354,7879543992,0,-1,2,0 days 00:00:00
1,1684893977,194554089,66838552,0,-1,2,0 days 00:00:00.000010735
2,1684893977,194564824,71733016,0,-1,2,0 days 00:00:00.000021470
3,1684893977,194574485,70979088,0,-1,2,0 days 00:00:00.000031131
4,1684893977,194585220,19186048,0,-1,2,0 days 00:00:00.000041866


In [ ]:
def get_combined_timestamp(row):
    return pd.to_datetime(datetime.datetime.fromtimestamp(row['unix_time_s']).strftime("%Y-%m-%dT%H:%M:%S") + "." + str(row['unix_time_ns']))

In [ ]:
# def time_difference(row, start_point):
#     '''
#     Time Difference between current element (row) and some previous point (start_point)
#     '''
#     row_s = row['unix_time_s']
#     row_ns = row['unix_time_ns']
#     if row['unix_time_ns'] < start_point['unix_time_ns']:
#         row_s -= 1
#         row_ns = 1e9 + row['unix_time_ns']
        
#     ns_diff = row_ns - start_point['unix_time_ns']
#     s_diff = row_s - start_point['unix_time_s']
#     print(s_diff, ": ", ns_diff)
#     return pd.to_datetime(datetime.datetime.fromtimestamp(s_diff).strftime("%Y-%m-%dT%H:%M:%S") + "." + str(ns_diff))

    
#     # return pd.to_datetime(s_diff + ns_diff)

In [ ]:
def get_datetime_min(time1, time2):
    time1_s, time1_ns = time1.unix_time_s, time1.unix_time_ns
    time2_s, time2_ns = time2.unix_time_s, time2.unix_time_ns
    
    if time1_s < time2_s:
        return time1
    elif time2_s < time1_s:
        return time2
    else:
        if time1_ns <= time2_ns:
            return time1
        return time2

## Save Files (Optional)

In [ ]:
# try:
#     os.mkdir(os.path.join(folder_path, 'modified'))
# except FileExistsError:
#     # print("already created")
#     pass

In [ ]:
# strg_read.to_csv(os.path.join(folder_path, 'modified', 'ata_read.csv'), index=False)
# strg_write.to_csv(os.path.join(folder_path, 'modified', 'ata_write.csv'), index=False)
# mem_read.to_csv(os.path.join(folder_path, 'modified', 'mem_read.csv'), index=False)
# mem_write.to_csv(os.path.join(folder_path, 'modified', 'mem_write.csv'), index=False)
# mem_read_write.to_csv(os.path.join(folder_path, 'modified', 'mem_readwrite.csv'), index=False)
# mem_exec.to_csv(os.path.join(folder_path, 'modified', 'mem_exec.csv'), index=False)

In [62]:
strg_read.head()

,unix_time_s,unix_time_ns,lba,storage_size,storage_entropy,relative_time
0,1684893978,693442531,38952048,4096,-1,0 days 00:00:01.498899177
1,1684893978,693442531,38952056,4096,-1,0 days 00:00:01.498899177
2,1684893978,693443604,38952064,4096,-1,0 days 00:00:01.498900250
3,1684893978,693443604,38952072,4096,-1,0 days 00:00:01.498900250
4,1684893978,693444678,38952080,4096,-1,0 days 00:00:01.498901324


In [63]:
strg_read_mod.head()

,unix_time_s,unix_time_ns,lba,storage_size,storage_entropy,action,timestamp,relative_time
0,1684893978,693442531,38952048,4096,-1,ata_read,2023-05-23 22:06:18.693442531,0 days 00:00:01.498899177
1,1684893978,693442531,38952056,4096,-1,ata_read,2023-05-23 22:06:18.693442531,0 days 00:00:01.498899177
2,1684893978,693443604,38952064,4096,-1,ata_read,2023-05-23 22:06:18.693443604,0 days 00:00:01.498900250
3,1684893978,693443604,38952072,4096,-1,ata_read,2023-05-23 22:06:18.693443604,0 days 00:00:01.498900250
4,1684893978,693444678,38952080,4096,-1,ata_read,2023-05-23 22:06:18.693444678,0 days 00:00:01.498901324


In [74]:
strg_read_file_path = 'RanSMAP\\dataset\\original\\i3-gen12\\ddr4-2133-16g\\AESCrypt\\AESCrypt-20230524_20-18-05\\new\\ada_read.pkl'
with open(strg_read_file_path, 'wb') as file:
    pickle.dump(strg_read, file)
    
strg_write_file_path = 'RanSMAP\\dataset\\original\\i3-gen12\\ddr4-2133-16g\\AESCrypt\\AESCrypt-20230524_20-18-05\\new\\ada_write.pkl'
with open(strg_write_file_path, 'wb') as file:
    pickle.dump(strg_write, file)

mem_read_file_path = 'RanSMAP\\dataset\\original\\i3-gen12\\ddr4-2133-16g\\AESCrypt\\AESCrypt-20230524_20-18-05\\new\\mem_read.pkl'
with open(mem_read_file_path, 'wb') as file:
    pickle.dump(mem_read, file)
    
mem_write_file_path = 'RanSMAP\\dataset\\original\\i3-gen12\\ddr4-2133-16g\\AESCrypt\\AESCrypt-20230524_20-18-05\\new\\mem_write.pkl'
with open(mem_write_file_path, 'wb') as file:
    pickle.dump(mem_write, file)

mem_read_write_file_path = 'RanSMAP\\dataset\\original\\i3-gen12\\ddr4-2133-16g\\AESCrypt\\AESCrypt-20230524_20-18-05\\new\\mem_readwrite.pkl'
with open(mem_read_write_file_path, 'wb') as file:
    pickle.dump(mem_read_write, file)

mem_exec_file_path = 'RanSMAP\\dataset\\original\\i3-gen12\\ddr4-2133-16g\\AESCrypt\\AESCrypt-20230524_20-18-05\\new\\mem_exec.pkl'
with open(mem_exec_file_path, 'wb') as file:
    pickle.dump(mem_exec, file)

##### tmp

In [105]:
strg_read.loc[0, ['unix_time_s', 'unix_time_ns']], strg_read.loc[4, ['unix_time_s', 'unix_time_ns']]

(unix_time_s     1684893978
 unix_time_ns     693442531
 Name: 0, dtype: object,
 unix_time_s     1684893978
 unix_time_ns     693444678
 Name: 4, dtype: object)

In [107]:
time_difference(strg_read.loc[4, ['unix_time_s', 'unix_time_ns']], strg_read.loc[0, ['unix_time_s', 'unix_time_ns']])

Timestamp('1970-01-01 00:00:00.000002147')

In [110]:
pd.to_datetime(strg_read.loc[4, 'unix_time_s'] + strg_read.loc[4, 'unix_time_ns']), pd.to_datetime(strg_read.loc[0, 'unix_time_s'] + strg_read.loc[0, 'unix_time_ns'])

(Timestamp('1970-01-01 00:00:02.378338656'),
 Timestamp('1970-01-01 00:00:02.378336509'))

In [114]:
strg_read.iloc[[0,1,2,3,4,95,96]]

,unix_time_s,unix_time_ns,lba,size,entropy,timestamp
0,1684893978,693442531,38952048,4096,-1,2023-05-23 22:06:18.693442531
1,1684893978,693442531,38952056,4096,-1,2023-05-23 22:06:18.693442531
2,1684893978,693443604,38952064,4096,-1,2023-05-23 22:06:18.693443604
3,1684893978,693443604,38952072,4096,-1,2023-05-23 22:06:18.693443604
4,1684893978,693444678,38952080,4096,-1,2023-05-23 22:06:18.693444678
95,1684893985,775139243,21854938,4096,-1,2023-05-23 22:06:25.775139243
96,1684893985,775139243,21854946,4096,-1,2023-05-23 22:06:25.775139243


In [123]:
strg_read.loc[len(strg_read)-1, 'unix_time_s']

np.int64(1684894177)

In [116]:
print(pd.to_datetime(strg_read.loc[0, 'unix_time_s'] + strg_read.loc[0, 'unix_time_ns']))
print(pd.to_datetime(strg_read.loc[2, 'unix_time_s'] + strg_read.loc[2, 'unix_time_ns']))
print(pd.to_datetime(strg_read.loc[96, 'unix_time_s'] + strg_read.loc[96, 'unix_time_ns']))

1970-01-01 00:00:02.378336509
1970-01-01 00:00:02.378337582
1970-01-01 00:00:02.460033228


In [125]:
pd.to_datetime(strg_read.loc[96, 'unix_time_s'] + strg_read.loc[96, 'unix_time_ns']) - pd.to_datetime(strg_read.loc[0, 'unix_time_s'] + strg_read.loc[0, 'unix_time_ns'])

Timedelta('0 days 00:00:00.081696719')

In [124]:
print(get_combined_timestamp(strg_read.loc[0, ['unix_time_s', 'unix_time_ns']]))
print(get_combined_timestamp(strg_read.loc[2, ['unix_time_s', 'unix_time_ns']]))
print(get_combined_timestamp(strg_read.loc[96, ['unix_time_s', 'unix_time_ns']]))
print(get_combined_timestamp(strg_read.loc[len(strg_read)-1, ['unix_time_s', 'unix_time_ns']]))

2023-05-23 22:06:18.693442531
2023-05-23 22:06:18.693443604
2023-05-23 22:06:25.775139243
2023-05-23 22:09:37.453444312


In [126]:
get_combined_timestamp(strg_read.loc[len(strg_read)-1, ['unix_time_s', 'unix_time_ns']]) - get_combined_timestamp(strg_read.loc[0, ['unix_time_s', 'unix_time_ns']])

Timedelta('0 days 00:03:18.760001781')

In [130]:
time_difference(strg_read.loc[len(strg_read)-1, ['unix_time_s', 'unix_time_ns']], strg_read.loc[0, ['unix_time_s', 'unix_time_ns']])

198 :  760001781.0


DateParseError: Unknown datetime string format, unable to parse: 1969-12-31T19:03:18.760001781.0, at position 0

In [83]:
strg_read['timestamp'] = strg_read.apply(lambda x: get_combined_timestamp(x), axis=1)

In [84]:
strg_read.head()

,unix_time_s,unix_time_ns,lba,size,entropy,timestamp
0,1684893978,693442531,38952048,4096,-1,2023-05-23 22:06:18.693442531
1,1684893978,693442531,38952056,4096,-1,2023-05-23 22:06:18.693442531
2,1684893978,693443604,38952064,4096,-1,2023-05-23 22:06:18.693443604
3,1684893978,693443604,38952072,4096,-1,2023-05-23 22:06:18.693443604
4,1684893978,693444678,38952080,4096,-1,2023-05-23 22:06:18.693444678


In [19]:
mem_exec.head()

,unix_time_s,unix_time_ns,gpa,size,entropy,mem_access_type
0,1684893977,194843934,43816432,0,-1,2
1,1684893977,194892241,44043172,0,-1,2
2,1684893977,194922299,52927184,0,-1,2
3,1684893977,198936116,19569910112,0,-1,2
4,1684893977,199392353,18447720484,0,-1,2


In [31]:
strg_read['timestamp'] = strg_read.apply(lambda x: datetime.datetime.fromtimestamp(x['unix_time_s'] + x['unix_time_ns'] / 1e9), axis=1)

In [ ]:
strg_read['timestamp'] = strg_read.apply(lambda x: datetime.datetime.fromtimestamp(x['unix_time_s'] + x['unix_time_ns'] / 1e9), axis=1)
strg_write['timestamp'] = strg_write.apply(lambda x: datetime.datetime.fromtimestamp(x['unix_time_s'] + x['unix_time_ns'] / 1e9), axis=1)
mem_read['timestamp'] = mem_read.apply(lambda x: datetime.datetime.fromtimestamp(x['unix_time_s'] + x['unix_time_ns'] / 1e9), axis=1)
mem_write['timestamp'] = mem_write.apply(lambda x: datetime.datetime.fromtimestamp(x['unix_time_s'] + x['unix_time_ns'] / 1e9), axis=1)
mem_read_write['timestamp'] = mem_read_write.apply(lambda x: datetime.datetime.fromtimestamp(x['unix_time_s'] + x['unix_time_ns'] / 1e9), axis=1)
mem_exec['timestamp'] = mem_exec.apply(lambda x: datetime.datetime.fromtimestamp(x['unix_time_s'] + x['unix_time_ns'] / 1e9), axis=1)


In [43]:
start_point = min(min(min(min(min(strg_read.loc[0, 'timestamp'], strg_write.loc[0, 'timestamp']), mem_read.loc[0, 'timestamp']), 
                          mem_write.loc[0, 'timestamp']), mem_read_write.loc[0, 'timestamp']), mem_exec.loc[0, 'timestamp'])

In [44]:
start_point

Timestamp('2023-05-23 22:06:17.194543')

In [45]:
strg_read['time_diff'] = strg_read['timestamp'].apply(lambda x: x-start_point)
strg_write['time_diff'] = strg_write['timestamp'].apply(lambda x: x-start_point)
mem_read['time_diff'] = mem_read['timestamp'].apply(lambda x: x-start_point)
mem_write['time_diff'] = mem_write['timestamp'].apply(lambda x: x-start_point)
mem_read_write['time_diff'] = mem_read_write['timestamp'].apply(lambda x: x-start_point)
mem_exec['time_diff'] = mem_exec['timestamp'].apply(lambda x: x-start_point)

In [49]:
strg_read.loc[0,'timestamp']

Timestamp('2023-05-23 22:06:18.693443')

In [46]:
strg_read.head()

,unix_time_s,unix_time_ns,lba,size,entropy,timestamp,time_diff
0,1684893978,693442531,38952048,4096,-1,2023-05-23 22:06:18.693443,0 days 00:00:01.498900
1,1684893978,693442531,38952056,4096,-1,2023-05-23 22:06:18.693443,0 days 00:00:01.498900
2,1684893978,693443604,38952064,4096,-1,2023-05-23 22:06:18.693444,0 days 00:00:01.498901
3,1684893978,693443604,38952072,4096,-1,2023-05-23 22:06:18.693444,0 days 00:00:01.498901
4,1684893978,693444678,38952080,4096,-1,2023-05-23 22:06:18.693445,0 days 00:00:01.498902


In [41]:
strg_read.loc[0, 'timestamp'], strg_write.loc[0, 'timestamp']

(Timestamp('2023-05-23 22:06:18.693443'),
 Timestamp('2023-05-23 22:06:19.562027'))

In [42]:
min(strg_read.loc[0, 'timestamp'], strg_write.loc[0, 'timestamp'])

Timestamp('2023-05-23 22:06:18.693443')

In [51]:
strg_read.head(100)

,unix_time_s,unix_time_ns,lba,size,entropy,timestamp,time_diff
0,1684893978,693442531,38952048,4096,-1,2023-05-23 22:06:18.693443,0 days 00:00:01.498900
1,1684893978,693442531,38952056,4096,-1,2023-05-23 22:06:18.693443,0 days 00:00:01.498900
2,1684893978,693443604,38952064,4096,-1,2023-05-23 22:06:18.693444,0 days 00:00:01.498901
3,1684893978,693443604,38952072,4096,-1,2023-05-23 22:06:18.693444,0 days 00:00:01.498901
4,1684893978,693444678,38952080,4096,-1,2023-05-23 22:06:18.693445,0 days 00:00:01.498902
...,...,...,...,...,...,...,...
95,1684893985,775139243,21854938,4096,-1,2023-05-23 22:06:25.775139,0 days 00:00:08.580596
96,1684893985,775139243,21854946,4096,-1,2023-05-23 22:06:25.775139,0 days 00:00:08.580596
97,1684893985,775140316,21854954,4096,-1,2023-05-23 22:06:25.775140,0 days 00:00:08.580597
98,1684893985,775141390,21854962,4096,-1,2023-05-23 22:06:25.775141,0 days 00:00:08.580598


In [ ]:
str

(447312, 454250, 173068)

In [63]:
pd.to_datetime(strg_read['unix_time_s'] + strg_read['unix_time_ns'])

0        1970-01-01 00:00:02.378336509
1        1970-01-01 00:00:02.378336509
2        1970-01-01 00:00:02.378337582
3        1970-01-01 00:00:02.378337582
4        1970-01-01 00:00:02.378338656
                      ...             
447307   1970-01-01 00:00:02.138338489
447308   1970-01-01 00:00:02.138338489
447309   1970-01-01 00:00:02.138338489
447310   1970-01-01 00:00:02.138338489
447311   1970-01-01 00:00:02.138338489
Length: 447312, dtype: datetime64[ns]

In [70]:
datetime.datetime.fromtimestamp(strg_read.loc[0, 'unix_time_s']).strftime("%Y-%m-%dT%H:%M:%S") + "." + str(strg_read.loc[0, 'unix_time_ns'])

'2023-05-23T22:06:18.693442531'

In [75]:
prev_pd_dt = pd.to_datetime(datetime.datetime.fromtimestamp(strg_read.loc[0, 'unix_time_s']).strftime("%Y-%m-%dT%H:%M:%S") + "." + str(strg_read.loc[0, 'unix_time_ns']))
prev_pd_dt

Timestamp('2023-05-23 22:06:18.693442531')

In [76]:
curr_pd_dt = pd.to_datetime(datetime.datetime.fromtimestamp(strg_read.loc[2, 'unix_time_s']).strftime("%Y-%m-%dT%H:%M:%S") + "." + str(strg_read.loc[2, 'unix_time_ns']))
curr_pd_dt

Timestamp('2023-05-23 22:06:18.693443604')

In [77]:
curr_pd_dt - prev_pd_dt

Timedelta('0 days 00:00:00.000001073')

In [65]:
strg_read.loc[0, 'unix_time_s'] + strg_read.loc[0, 'unix_time_ns']

np.int64(2378336509)

In [61]:
print(strg_read.loc[0, 'unix_time_s'], ": ", strg_read.loc[0, 'unix_time_ns'])
print(strg_read.loc[2, 'unix_time_s'], ": ", strg_read.loc[2, 'unix_time_ns'])

1684893978 :  693442531
1684893978 :  693443604


In [58]:
prev_total_ns = strg_read.loc[0, 'unix_time_s'] * 1e9 + strg_read.loc[0, 'unix_time_ns']
current_total_ns = strg_read.loc[2, 'unix_time_s'] * 1e9 + strg_read.loc[2, 'unix_time_ns']

diff_ns = current_total_ns - prev_total_ns
diff_seconds = int(diff_ns // 1e9)
diff_nanoseconds = int(diff_ns % 1e9)

diff_ns, diff_seconds, diff_nanoseconds

(np.float64(1024.0), 0, 1024)

In [60]:
current_total_ns, prev_total_ns

(np.float64(1.6848939786934436e+18), np.float64(1.6848939786934426e+18))

In [59]:
3604-2531

1073

(np.float64(7081698816.0), 7, 81698816)

In [ ]:
def compare(time1_s, time1_ns, time2_s, time2_ns):
    if time1_s < time2_s:
        return True
    elif time2_s < time1_s:
        return False
    else:
        return time1_ns < time2_ns

In [ ]:
# Combine unix_time_s and unix_time_ns into a single timestamp
# Convert nanoseconds to seconds by dividing by 1e9
combined_timestamp = mem_exec['unix_time_s'] + mem_exec['unix_time_ns'] / 1e9

# Display the first few combined timestamps
print("Combined timestamps (seconds since Unix epoch):")
print(combined_timestamp.head())

# If you want to convert to human-readable datetime
import datetime
readable_dates = [datetime.datetime.fromtimestamp(ts) for ts in combined_timestamp.head()]
print("\nHuman-readable dates:")
for date in readable_dates:
    print(date)

In [21]:
mem_exec.loc[0, 'unix_time_s'] + mem_exec.loc[0, 'unix_time_ns'] / 1e9

np.float64(1684893977.194844)

In [35]:
datetime.datetime.fromtimestamp(mem_exec.loc[0, 'unix_time_s'] + mem_exec.loc[0, 'unix_time_ns'] / 1e9), datetime.datetime.fromtimestamp(strg_read.loc[1, 'unix_time_s'] + strg_read.loc[1, 'unix_time_ns'] / 1e9)

(datetime.datetime(2023, 5, 23, 22, 6, 17, 194844),
 datetime.datetime(2023, 5, 23, 22, 6, 18, 693443))

In [24]:
datetime.datetime.fromtimestamp(mem_exec.loc[0, 'unix_time_s'] + mem_exec.loc[0, 'unix_time_ns'] / 1e9), datetime.datetime.fromtimestamp(mem_exec.loc[1, 'unix_time_s'] + mem_exec.loc[1, 'unix_time_ns'] / 1e9)

(datetime.datetime(2023, 5, 23, 22, 6, 17, 194844),
 datetime.datetime(2023, 5, 23, 22, 6, 17, 194892))

In [34]:
datetime.datetime.fromtimestamp(mem_exec.loc[0, 'unix_time_s'] + mem_exec.loc[0, 'unix_time_ns'] / 1e8), datetime.datetime.fromtimestamp(mem_exec.loc[1, 'unix_time_s'] + mem_exec.loc[1, 'unix_time_ns'] / 1e9)

(datetime.datetime(2023, 5, 23, 22, 6, 18, 948439),
 datetime.datetime(2023, 5, 23, 22, 6, 17, 194892))

In [37]:
mem_exec.head()

,unix_time_s,unix_time_ns,gpa,size,entropy,mem_access_type
0,1684893977,194843934,43816432,0,-1,2
1,1684893977,194892241,44043172,0,-1,2
2,1684893977,194922299,52927184,0,-1,2
3,1684893977,198936116,19569910112,0,-1,2
4,1684893977,199392353,18447720484,0,-1,2


In [36]:
datetime.datetime.fromtimestamp(mem_exec.loc[1, 'unix_time_s'] + mem_exec.loc[1, 'unix_time_ns'] / 1e9) - datetime.datetime.fromtimestamp(mem_exec.loc[0, 'unix_time_s'] + mem_exec.loc[0, 'unix_time_ns'] / 1e9)

datetime.timedelta(microseconds=48)

In [38]:
92241 - 43934

48307

## Group Tables Together

In [75]:
strg_comb = pd.concat([strg_read, strg_write], ignore_index=True)

In [76]:
strg_comb.drop(['unix_time_s', 'unix_time_ns'], axis=1, inplace=True)

In [77]:
mem_comb = pd.concat([mem_read, mem_write, mem_read_write, mem_exec], ignore_index=True)

In [78]:
mem_comb.drop(['unix_time_s', 'unix_time_ns'], axis=1, inplace=True)

In [108]:
tmp_comb_new = pd.concat([strg_read_new, strg_write_new, mem_read_new, mem_write_new, mem_read_write_new, mem_exec_new], ignore_index=True)
tmp_comb_new.drop(['unix_time_s', 'unix_time_ns'], axis=1, inplace=True)

In [79]:
strg_comb.head()

,lba,storage_size,storage_entropy,relative_time,action
0,38952048,4096,-1.0,0 days 00:00:01.498899177,ata_read
1,38952056,4096,-1.0,0 days 00:00:01.498899177,ata_read
2,38952064,4096,-1.0,0 days 00:00:01.498900250,ata_read
3,38952072,4096,-1.0,0 days 00:00:01.498900250,ata_read
4,38952080,4096,-1.0,0 days 00:00:01.498901324,ata_read


In [80]:
mem_comb.head()

,gpa,mem_size,mem_entropy,mem_access_type,relative_time,action
0,7879543992,0,-1.0,2,0 days 00:00:00,mem_read
1,66838552,0,-1.0,2,0 days 00:00:00.000010735,mem_read
2,71733016,0,-1.0,2,0 days 00:00:00.000021470,mem_read
3,70979088,0,-1.0,2,0 days 00:00:00.000031131,mem_read
4,19186048,0,-1.0,2,0 days 00:00:00.000041866,mem_read


In [81]:
tmp_comb = pd.concat([strg_comb, mem_comb], ignore_index=True)

In [82]:
tmp_comb.head()

,lba,storage_size,storage_entropy,relative_time,action,gpa,mem_size,mem_entropy,mem_access_type
0,38952048.0,4096.0,-1.0,0 days 00:00:01.498899177,ata_read,NaN,NaN,NaN,NaN
1,38952056.0,4096.0,-1.0,0 days 00:00:01.498899177,ata_read,NaN,NaN,NaN,NaN
2,38952064.0,4096.0,-1.0,0 days 00:00:01.498900250,ata_read,NaN,NaN,NaN,NaN
3,38952072.0,4096.0,-1.0,0 days 00:00:01.498900250,ata_read,NaN,NaN,NaN,NaN
4,38952080.0,4096.0,-1.0,0 days 00:00:01.498901324,ata_read,NaN,NaN,NaN,NaN


In [83]:
tmp_comb.tail()

,lba,storage_size,storage_entropy,relative_time,action,gpa,mem_size,mem_entropy,mem_access_type
1254926,NaN,NaN,NaN,0 days 00:00:54.950592507,mem_exec,8.127819e+09,0.0,-1.0,2.0
1254927,NaN,NaN,NaN,0 days 00:00:54.951887148,mem_exec,8.161414e+09,0.0,-1.0,2.0
1254928,NaN,NaN,NaN,0 days 00:00:54.960752111,mem_exec,8.127898e+09,0.0,-1.0,2.0
1254929,NaN,NaN,NaN,0 days 00:01:12.830165229,mem_exec,4.364575e+09,0.0,-1.0,2.0
1254930,NaN,NaN,NaN,0 days 00:02:54.973250844,mem_exec,1.109762e+10,0.0,-1.0,2.0


In [109]:
tmp_comb_new.head()

,lba,storage_size,storage_entropy,action,relative_time,gpa,mem_size,mem_entropy,mem_access_type
0,38952048.0,4096.0,-1.0,ata_read,0 days 00:00:01.498899177,NaN,NaN,NaN,NaN
1,38952056.0,4096.0,-1.0,ata_read,0 days 00:00:01.498899177,NaN,NaN,NaN,NaN
2,38952064.0,4096.0,-1.0,ata_read,0 days 00:00:01.498900250,NaN,NaN,NaN,NaN
3,38952072.0,4096.0,-1.0,ata_read,0 days 00:00:01.498900250,NaN,NaN,NaN,NaN
4,38952080.0,4096.0,-1.0,ata_read,0 days 00:00:01.498901324,NaN,NaN,NaN,NaN


In [110]:
tmp_comb_new.tail()

,lba,storage_size,storage_entropy,action,relative_time,gpa,mem_size,mem_entropy,mem_access_type
1254926,NaN,NaN,NaN,mem_exec,0 days 00:00:54.950592507,8.127819e+09,0.0,-1.0,2.0
1254927,NaN,NaN,NaN,mem_exec,0 days 00:00:54.951887148,8.161414e+09,0.0,-1.0,2.0
1254928,NaN,NaN,NaN,mem_exec,0 days 00:00:54.960752111,8.127898e+09,0.0,-1.0,2.0
1254929,NaN,NaN,NaN,mem_exec,0 days 00:01:12.830165229,4.364575e+09,0.0,-1.0,2.0
1254930,NaN,NaN,NaN,mem_exec,0 days 00:02:54.973250844,1.109762e+10,0.0,-1.0,2.0


In [195]:
tmp_df = pd.DataFrame(range(0,100), index=pd.TimedeltaIndex(range(0,100),'s'))

C:\Users\malio\AppData\Local\Temp\ipykernel_24068\2412460305.py:1: FutureWarning: The 'unit' keyword in TimedeltaIndex construction is deprecated and will be removed in a future version. Use pd.to_timedelta instead.
  tmp_df = pd.DataFrame(range(0,100), index=pd.TimedeltaIndex(range(0,100),'s'))


In [200]:
tmp_df.head(20)

,0
0 days 00:00:00,0
0 days 00:00:01,1
0 days 00:00:02,2
0 days 00:00:03,3
0 days 00:00:04,4
0 days 00:00:05,5
0 days 00:00:06,6
0 days 00:00:07,7
0 days 00:00:08,8
0 days 00:00:09,9


In [201]:
tmp_df.rolling('10s').mean().head(20)#.resample('5s').first()

,0
0 days 00:00:00,0.0
0 days 00:00:01,0.5
0 days 00:00:02,1.0
0 days 00:00:03,1.5
0 days 00:00:04,2.0
0 days 00:00:05,2.5
0 days 00:00:06,3.0
0 days 00:00:07,3.5
0 days 00:00:08,4.0
0 days 00:00:09,4.5


In [202]:
tmp_df.rolling('10s').mean().resample('5s').first()

,0
0 days 00:00:00,0.0
0 days 00:00:05,2.5
0 days 00:00:10,5.5
0 days 00:00:15,10.5
0 days 00:00:20,15.5
0 days 00:00:25,20.5
0 days 00:00:30,25.5
0 days 00:00:35,30.5
0 days 00:00:40,35.5
0 days 00:00:45,40.5


In [205]:
strg_read[strg_read['action'] == 'ata_read']

,unix_time_s,unix_time_ns,lba,storage_size,storage_entropy,action,timestamp,relative_time
0,1684893978,693442531,38952048,4096,-1,ata_read,2023-05-23 22:06:18.693442531,0 days 00:00:01.498899177
1,1684893978,693442531,38952056,4096,-1,ata_read,2023-05-23 22:06:18.693442531,0 days 00:00:01.498899177
2,1684893978,693443604,38952064,4096,-1,ata_read,2023-05-23 22:06:18.693443604,0 days 00:00:01.498900250
3,1684893978,693443604,38952072,4096,-1,ata_read,2023-05-23 22:06:18.693443604,0 days 00:00:01.498900250
4,1684893978,693444678,38952080,4096,-1,ata_read,2023-05-23 22:06:18.693444678,0 days 00:00:01.498901324
...,...,...,...,...,...,...,...,...
447307,1684894177,453444312,120561480,4096,-1,ata_read,2023-05-23 22:09:37.453444312,0 days 00:03:20.258900958
447308,1684894177,453444312,120561488,4096,-1,ata_read,2023-05-23 22:09:37.453444312,0 days 00:03:20.258900958
447309,1684894177,453444312,120561496,4096,-1,ata_read,2023-05-23 22:09:37.453444312,0 days 00:03:20.258900958
447310,1684894177,453444312,120561504,4096,-1,ata_read,2023-05-23 22:09:37.453444312,0 days 00:03:20.258900958


## Generate Features

In [229]:
tmp_comb[tmp_comb['action']=='ata_read']['storage_size'].sum()

np.float64(1826286592.0)

In [14]:
def ransmap_preprocessing(df: pd.DataFrame, is_ransomware: bool, operation: str, window_s: int, epsilon_s: int = 1) -> pd.DataFrame:
    '''
    window_s: Window size in seconds
    epsilon_s: Step size in seconds
    '''
    
    preprocessed_df = pd.DataFrame(columns=["start", "end", 
                                            "avg_ata_read", "avg_ata_write", "lba_read_var", "lba_write_var", "avg_storage_entropy", 
                                            "avg_entropy_mem_write", "avg_entropy_mem_read_write", 
                                            "num_4KB_pages_mem_read", "num_4KB_pages_mem_write", "num_4KB_pages_mem_read_write", "num_4KB_pages_mem_exec",
                                            "num_2MB_pages_mem_read", "num_2MB_pages_mem_write", "num_2MB_pages_mem_read_write", "num_2MB_pages_mem_exec",
                                            "num_MIMO_pages_mem_read", "num_MIMO_pages_mem_write", "num_MIMO_pages_mem_read_write", "num_MIMO_pages_mem_exec",
                                            "gpa_mem_read_var", "gpa_mem_write_var", "gpa_mem_read_write_var", "gpa_mem_exec_var",
                                            ])
    # preprocessed_df = pd.DataFrame(columns=["start", "end", "avg_read"])
    window_delta = datetime.timedelta(seconds=window_s)
    epsilon_delta = datetime.timedelta(seconds=epsilon_s)
    
    last_start_possible = max(df['relative_time']) - window_delta
        
    start_time = min(df['relative_time']) # should be 0s
    while start_time <= last_start_possible:
        end_time = start_time + window_delta
        d_ata_read = df[(start_time <= df['relative_time']) & (df['relative_time'] <= end_time) & (df['action'] == 'ata_read')][["lba", "storage_size"]]
        d_ata_write = df[(start_time <= df['relative_time']) & (df['relative_time'] <= end_time) & (df['action'] == 'ata_write')][["lba", "storage_size", "storage_entropy"]]
        d_mem_read = df[(start_time <= df['relative_time']) & (df['relative_time'] <= end_time) & (df['action'] == 'mem_read')][["gpa", 'mem_size', 'mem_entropy', 'mem_access_type']]
        d_mem_write = df[(start_time <= df['relative_time']) & (df['relative_time'] <= end_time) & (df['action'] == 'mem_write')][["gpa", 'mem_size', 'mem_entropy', 'mem_access_type']]
        d_mem_read_write = df[(start_time <= df['relative_time']) & (df['relative_time'] <= end_time) & (df['action'] == 'mem_read_write')][["gpa", 'mem_size', 'mem_entropy', 'mem_access_type']]
        d_mem_exec = df[(start_time <= df['relative_time']) & (df['relative_time'] <= end_time) & (df['action'] == 'mem_exec')][["gpa", 'mem_size', 'mem_entropy', 'mem_access_type']]
        
        new = [start_time, end_time, 
               d_ata_read['storage_size'].sum() / window_s,
               d_ata_write['storage_size'].sum() / window_s,
               d_ata_read['lba'].var(),
               d_ata_write['lba'].var(),
               d_ata_write['storage_entropy'].mean(),
               
               d_mem_write['mem_entropy'].mean(), d_mem_read_write['mem_entropy'].mean(),
               len(d_mem_read[d_mem_read['mem_access_type'] == 1]), len(d_mem_write[d_mem_write['mem_access_type'] == 1]), len(d_mem_read_write[d_mem_read_write['mem_access_type'] == 1]), len(d_mem_exec[d_mem_exec['mem_access_type'] == 1]),
               len(d_mem_read[d_mem_read['mem_access_type'] == 2]), len(d_mem_write[d_mem_write['mem_access_type'] == 2]), len(d_mem_read_write[d_mem_read_write['mem_access_type'] == 2]), len(d_mem_exec[d_mem_exec['mem_access_type'] == 2]),
               len(d_mem_read[d_mem_read['mem_access_type'] == 4]), len(d_mem_write[d_mem_write['mem_access_type'] == 4]), len(d_mem_read_write[d_mem_read_write['mem_access_type'] == 4]), len(d_mem_exec[d_mem_exec['mem_access_type'] == 4]),
               d_mem_read['gpa'].var(), d_mem_write['gpa'].var(), d_mem_read_write['gpa'].var(), d_mem_exec['gpa'].var()
               ]
        
        start_time += epsilon_delta
        preprocessed_df.loc[len(preprocessed_df)] = new

    preprocessed_df['is_ransomware'] = is_ransomware
    preprocessed_df['operation'] = operation
    return preprocessed_df

In [111]:
tmp_prep_df = ransmap_preprocessing(tmp_comb_new, False, 'AESCrypt', 30, 1)

In [114]:
datetime.timedelta(seconds=1)

datetime.timedelta(seconds=1)

In [115]:
mem_read_new[mem_read_new['mem_access_type'] == 4]

,unix_time_s,unix_time_ns,gpa,mem_size,mem_entropy,mem_access_type,action,relative_time
365,1684893977,512196298,1887721504,0,-1,4,mem_read,0 days 00:00:00.317652944
366,1684893977,512220988,1887698952,0,-1,4,mem_read,0 days 00:00:00.317677634
367,1684893977,512246752,1887722320,0,-1,4,mem_read,0 days 00:00:00.317703398
368,1684893977,512320824,1887722320,0,-1,4,mem_read,0 days 00:00:00.317777470
369,1684893977,512346588,1887722320,0,-1,4,mem_read,0 days 00:00:00.317803234
...,...,...,...,...,...,...,...,...
173063,1684894177,453051411,1889674036,0,-1,4,mem_read,0 days 00:03:20.258508057
173064,1684894177,453500134,1889673224,0,-1,4,mem_read,0 days 00:03:20.258956780
173065,1684894177,453514089,1889674000,0,-1,4,mem_read,0 days 00:03:20.258970735
173066,1684894177,453529118,1889674040,0,-1,4,mem_read,0 days 00:03:20.258985764


In [117]:
mem_read_new[(mem_read_new['mem_access_type'] == 4) & (datetime.timedelta(seconds=1) <= mem_read_new['relative_time']) & (mem_read_new['relative_time'] <= datetime.timedelta(seconds=31))]

,unix_time_s,unix_time_ns,gpa,mem_size,mem_entropy,mem_access_type,action,relative_time
727,1684893978,528662428,1887721504,0,-1,4,mem_read,0 days 00:00:01.334119074
728,1684893978,528672089,1887698952,0,-1,4,mem_read,0 days 00:00:01.334128735
729,1684893978,528724691,1887722320,0,-1,4,mem_read,0 days 00:00:01.334181337
730,1684893978,528798762,1887722320,0,-1,4,mem_read,0 days 00:00:01.334255408
731,1684893978,528824526,1887722320,0,-1,4,mem_read,0 days 00:00:01.334281172
...,...,...,...,...,...,...,...,...
20684,1684894008,122334083,1889674040,0,-1,4,mem_read,0 days 00:00:30.927790729
20685,1684894008,122389905,1889674036,0,-1,4,mem_read,0 days 00:00:30.927846551
20686,1684894008,122597090,1889673224,0,-1,4,mem_read,0 days 00:00:30.928053736
20687,1684894008,122609972,1889674000,0,-1,4,mem_read,0 days 00:00:30.928066618


In [112]:
tmp_prep_df.head()

,start,end,avg_ata_read,avg_ata_write,lba_read_var,lba_write_var,avg_storage_entropy,avg_entropy_mem_write,avg_entropy_mem_read_write,num_4KB_pages_mem_read,...,num_MIMO_pages_mem_read,num_MIMO_pages_mem_write,num_MIMO_pages_mem_read_write,num_MIMO_pages_mem_exec,gpa_mem_read_var,gpa_mem_write_var,gpa_mem_read_write_var,gpa_mem_exec_var,is_ransomware,operation
0,0 days 00:00:00,0 days 00:00:30,2.218035e+06,156074.666667,6.906892e+14,3.467378e+15,0.339909,0.008179,0.757494,10,...,18824,19116,0,0,6.087713e+18,1.611204e+19,3.653646e+19,3.859055e+19,False,AESCrypt
1,0 days 00:00:01,0 days 00:00:31,2.369860e+06,287010.133333,6.640260e+14,2.397020e+15,0.356280,0.006177,0.625245,3,...,19660,19955,0,0,6.797337e+17,3.460737e+17,2.335112e+19,2.666074e+19,False,AESCrypt
2,0 days 00:00:02,0 days 00:00:32,2.365577e+06,302028.800000,6.595067e+14,2.377311e+15,0.361597,0.005718,0.689899,2,...,19799,20092,0,0,3.764257e+17,2.918401e+17,2.929815e+19,4.008223e+18,False,AESCrypt
3,0 days 00:00:03,0 days 00:00:33,2.365577e+06,277725.866667,6.595067e+14,2.243987e+15,0.325978,0.005725,0.689899,2,...,19641,19932,0,0,3.794003e+17,2.925154e+17,2.929815e+19,4.008223e+18,False,AESCrypt
4,0 days 00:00:04,0 days 00:00:34,2.364621e+06,277179.733333,6.597213e+14,2.245231e+15,0.325634,0.005700,0.689899,2,...,19626,19918,0,0,3.813220e+17,2.928641e+17,2.929815e+19,4.008223e+18,False,AESCrypt


In [269]:
tmp_prep_df

,start,end,avg_ata_read,avg_ata_write,lba_read_var,lba_write_var,avg_storage_entropy,avg_entropy_mem_write,avg_entropy_mem_read_write,num_4KB_pages_mem_read,...,num_2MB_pages_mem_read_write,num_2MB_pages_mem_exec,num_MIMO_pages_mem_read,num_MIMO_pages_mem_write,num_MIMO_pages_mem_read_write,num_MIMO_pages_mem_exec,gpa_mem_read_var,gpa_mem_write_var,gpa_mem_read_write_var,gpa_mem_exec_var
0,0 days 00:00:00,0 days 00:00:30,2.218035e+06,1.560747e+05,6.906892e+14,3.467378e+15,0.339909,0.008179,0.757494,10,...,82,250,18824,19116,0,0,6.087713e+18,1.611204e+19,3.653646e+19,3.859055e+19
1,0 days 00:00:01,0 days 00:00:31,2.368085e+06,2.147840e+05,6.621935e+14,3.003999e+15,0.352099,0.006297,0.625245,3,...,22,30,19277,19572,0,0,6.928751e+17,3.527299e+17,2.335112e+19,2.666074e+19
2,0 days 00:00:02,0 days 00:00:32,2.365577e+06,3.006635e+05,6.595067e+14,2.374801e+15,0.360908,0.005721,0.689899,2,...,7,13,19789,20080,0,0,3.766126e+17,2.920116e+17,2.929815e+19,4.008223e+18
3,0 days 00:00:03,0 days 00:00:33,2.365577e+06,2.777259e+05,6.595067e+14,2.243987e+15,0.325978,0.005725,0.689899,2,...,7,13,19641,19932,0,0,3.794003e+17,2.925154e+17,2.929815e+19,4.008223e+18
4,0 days 00:00:04,0 days 00:00:34,2.364621e+06,2.771797e+05,6.597213e+14,2.245231e+15,0.325634,0.005700,0.689899,2,...,7,13,19626,19918,0,0,3.813220e+17,2.928641e+17,2.929815e+19,4.008223e+18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,0 days 00:02:46,0 days 00:03:16,9.190537e+06,9.819204e+06,3.632157e+14,9.309250e+14,0.962230,0.028495,0.852640,0,...,124,1,23868,24252,0,0,2.595691e+18,2.986827e+18,2.468795e+18,NaN
167,0 days 00:02:47,0 days 00:03:17,8.961570e+06,9.750938e+06,3.632172e+14,9.346152e+14,0.961998,0.028067,0.848693,0,...,113,1,23504,23889,0,0,2.554711e+18,2.972694e+18,2.684153e+18,NaN
168,0 days 00:02:48,0 days 00:03:18,9.041818e+06,1.017544e+07,3.652167e+14,9.319532e+14,0.961042,0.027519,0.845694,0,...,118,1,24150,24492,0,0,2.604059e+18,2.937151e+18,2.569516e+18,NaN
169,0 days 00:02:49,0 days 00:03:19,9.162735e+06,1.027238e+07,3.637669e+14,9.290462e+14,0.960527,0.027545,0.845694,0,...,118,1,24457,24831,0,0,2.604661e+18,2.931009e+18,2.569516e+18,NaN


In [278]:
tmp_prep_df[[col for col in tmp_prep_df.columns.to_list() if 'MIMO' in col]].max()

num_MIMO_pages_mem_read          36268
num_MIMO_pages_mem_write         37667
num_MIMO_pages_mem_read_write        0
num_MIMO_pages_mem_exec              0
dtype: int64

In [243]:
tmp_comb[(tmp_prep_df.loc[0, 'start'] <= tmp_comb['relative_time']) & (tmp_comb['relative_time'] <= tmp_prep_df.loc[0, 'end']) & (tmp_comb['action'] == 'ata_read')][['relative_time', 'storage_size']]

,relative_time,storage_size
0,0 days 00:00:01.498899177,4096.0
1,0 days 00:00:01.498899177,4096.0
2,0 days 00:00:01.498900250,4096.0
3,0 days 00:00:01.498900250,4096.0
4,0 days 00:00:01.498901324,4096.0
...,...,...
16326,0 days 00:00:28.092173423,4096.0
16327,0 days 00:00:28.092174496,4096.0
16328,0 days 00:00:28.092174496,4096.0
16329,0 days 00:00:28.092175570,4096.0


# Final Function

In [118]:
# def get_combined_timestamp(row):
#     return pd.to_datetime(datetime.datetime.fromtimestamp(row['unix_time_s']).strftime("%Y-%m-%dT%H:%M:%S") + "." + str(row['unix_time_ns']))

def get_datetime_min(time1, time2):
    time1_s, time1_ns = time1.unix_time_s, time1.unix_time_ns
    time2_s, time2_ns = time2.unix_time_s, time2.unix_time_ns
    
    if time1_s < time2_s:
        return time1
    elif time2_s < time1_s:
        return time2
    else:
        if time1_ns <= time2_ns:
            return time1
        return time2

def compute_relative_time(df, start_sec, start_nsec):
    """
    Compute relative time (in seconds and nanoseconds) from a start point.
    Keeps full nanosecond precision.
    """
    # Convert everything to total nanoseconds from the start point
    total_ns = (df['unix_time_s'] - start_sec) * 1_000_000_000 + (df['unix_time_ns'] - start_nsec)

    df['relative_time'] = pd.to_timedelta(total_ns, unit='ns')
    return df

In [ ]:
# def read_access_file(operation_path):
#     folder_paths = [os.path.join(operation_path, os.listdir(operation_path)[0])]
#     strg_read = pd.DataFrame(columns = strg_access_column_names)
#     strg_write = pd.DataFrame(columns = strg_access_column_names)
#     mem_read = pd.DataFrame(columns = mem_access_columns_names)
#     mem_write = pd.DataFrame(columns = mem_access_columns_names)
#     mem_read_write = pd.DataFrame(columns = mem_access_columns_names)
#     mem_exec = pd.DataFrame(columns = mem_access_columns_names)
    
#     for folder_path in folder_paths:
#         strg_read = pd.concat([strg_read, pd.read_csv(os.path.join(folder_path, 'ata_read.csv'), names = strg_access_column_names, usecols=[i for i in range(5)])])
#         strg_write = pd.concat([strg_write, pd.read_csv(os.path.join(folder_path, 'ata_write.csv'), names = strg_access_column_names, usecols=[i for i in range(5)])])
#         mem_read = pd.concat([mem_read, pd.read_csv(os.path.join(folder_path, 'mem_read.csv'), names = mem_access_columns_names)])
#         mem_write = pd.concat([mem_write, pd.read_csv(os.path.join(folder_path, 'mem_write.csv'), names = mem_access_columns_names)])
#         mem_read_write = pd.concat([mem_read_write, pd.read_csv(os.path.join(folder_path, 'mem_readwrite.csv'), names = mem_access_columns_names)])
#         mem_exec = pd.concat([mem_exec, pd.read_csv(os.path.join(folder_path, 'mem_exec.csv'), names = mem_access_columns_names)])
    
#     return strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec

In [10]:
kaggle_files_to_load

{'ata_read': {0: 'unix_time_s',
  1: 'unix_time_ns',
  2: 'lba',
  3: 'storage_size',
  4: 'storage_entropy'},
 'ata_write': {0: 'unix_time_s',
  1: 'unix_time_ns',
  2: 'lba',
  3: 'storage_size',
  4: 'storage_entropy'},
 'mem_read': {0: 'unix_time_s',
  1: 'unix_time_ns',
  2: 'gpa',
  3: 'mem_size',
  4: 'mem_entropy',
  5: 'mem_access_type'},
 'mem_write': {0: 'unix_time_s',
  1: 'unix_time_ns',
  2: 'gpa',
  3: 'mem_size',
  4: 'mem_entropy',
  5: 'mem_access_type'},
 'mem_readwrite': {0: 'unix_time_s',
  1: 'unix_time_ns',
  2: 'gpa',
  3: 'mem_size',
  4: 'mem_entropy',
  5: 'mem_access_type'},
 'mem_exec': {0: 'unix_time_s',
  1: 'unix_time_ns',
  2: 'gpa',
  3: 'mem_size',
  4: 'mem_entropy',
  5: 'mem_access_type'}}

In [12]:
def read_kaggle_access_file(HWR_PATH, DATASET, operations_logs, kaggle_files_to_load):
    strg_read = None #pd.DataFrame(columns = strg_access_column_names)
    strg_write = None #pd.DataFrame(columns = strg_access_column_names)
    mem_read = None #pd.DataFrame(columns = mem_access_columns_names)
    mem_write = None #pd.DataFrame(columns = mem_access_columns_names)
    mem_read_write = None #pd.DataFrame(columns = mem_access_columns_names)
    mem_exec = None #pd.DataFrame(columns = mem_access_columns_names)
    
            
    for file, columns in kaggle_files_to_load.items():
        comb_df = pd.DataFrame()
        for log in operations_logs:
            file_path = f"{HWR_PATH}/{log}/{file}.csv"
            new_df = kagglehub.dataset_load(
                KaggleDatasetAdapter.PANDAS,
                DATASET,
                file_path,
                pandas_kwargs={"names": columns.values(), "usecols": columns.keys()}
                )
            
            comb_df = pd.concat([comb_df, new_df], ignore_index=True)
            
        match file:
            case 'ata_read':
                # strg_read = pd.concat([strg_read, new_df], ignore_index=True)
                strg_read = comb_df
            case 'ata_write':
                # strg_write = pd.concat([strg_write, new_df], ignore_index=True)
                strg_write = comb_df
            case 'mem_read':
                # mem_read = pd.concat([mem_read, new_df], ignore_index=True)
                mem_read = comb_df
            case 'mem_write':
                # mem_write = pd.concat([mem_write, new_df], ignore_index=True)
                mem_write = comb_df
            case 'mem_readwrite':
                # mem_read_write = pd.concat([mem_read_write, new_df], ignore_index=True)
                mem_read_write = comb_df
            case 'mem_exec':
                # mem_exec = pd.concat([mem_exec, new_df], ignore_index=True)
                mem_exec = comb_df
                        
    
    return strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec

In [119]:
def generate_combined_log_file(strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec):
    
    # strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec = read_access_file(operation_path)
    
    strg_read['action'] = 'ata_read'
    strg_write['action'] = 'ata_write'
    mem_read['action'] = 'mem_read'
    mem_write['action'] = 'mem_write'
    mem_read_write['action'] = 'mem_read_write'
    mem_exec['action'] = 'mem_exec'
    
    start_point = get_datetime_min(
                    get_datetime_min(
                        get_datetime_min(
                            get_datetime_min(
                                get_datetime_min(strg_read.loc[0, ['unix_time_s', 'unix_time_ns']], strg_write.loc[0, ['unix_time_s', 'unix_time_ns']]), 
                                mem_read.loc[0, ['unix_time_s', 'unix_time_ns']]), 
                            mem_write.loc[0, ['unix_time_s', 'unix_time_ns']]), 
                        mem_read_write.loc[0, ['unix_time_s', 'unix_time_ns']]), 
                    mem_exec.loc[0, ['unix_time_s', 'unix_time_ns']])

    # start_point_datetime = get_combined_timestamp(start_point)
    
    # strg_read['timestamp'] = strg_read[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
    # strg_write['timestamp'] = strg_write[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
    # mem_read['timestamp'] = mem_read[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
    # mem_write['timestamp'] = mem_write[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
    # mem_read_write['timestamp'] = mem_read_write[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
    # mem_exec['timestamp'] = mem_exec[['unix_time_s', 'unix_time_ns']].apply(lambda x: get_combined_timestamp(x), axis=1)
    
    # strg_read['relative_time'] = strg_read['timestamp'].apply(lambda x: x - start_point_datetime)
    # strg_write['relative_time'] = strg_write['timestamp'].apply(lambda x: x - start_point_datetime)
    # mem_read['relative_time'] = mem_read[['timestamp']].apply(lambda x: x - start_point_datetime)
    # mem_write['relative_time'] = mem_write[['timestamp']].apply(lambda x: x - start_point_datetime)
    # mem_read_write['relative_time'] = mem_read_write[['timestamp']].apply(lambda x: x - start_point_datetime)
    # mem_exec['relative_time'] = mem_exec[['timestamp']].apply(lambda x: x - start_point_datetime)
    
    # start_sec, start_nsec = start_point.unix_time_s
    strg_read = compute_relative_time(strg_read, start_point.unix_time_s, start_point.unix_time_ns)
    strg_write = compute_relative_time(strg_write, start_point.unix_time_s, start_point.unix_time_ns)
    mem_read = compute_relative_time(mem_read, start_point.unix_time_s, start_point.unix_time_ns)
    mem_write = compute_relative_time(mem_write, start_point.unix_time_s, start_point.unix_time_ns)
    mem_read_write = compute_relative_time(mem_read_write, start_point.unix_time_s, start_point.unix_time_ns)
    mem_exec = compute_relative_time(mem_exec, start_point.unix_time_s, start_point.unix_time_ns)

    # strg_comb = pd.concat([strg_read, strg_write], ignore_index=True)
    # strg_comb.drop(['unix_time_s', 'unix_time_ns'], axis=1, inplace=True)
    
    # mem_comb = pd.concat([mem_read, mem_write, mem_read_write, mem_exec], ignore_index=True)
    # mem_comb.drop(['unix_time_s', 'unix_time_ns'], axis=1, inplace=True)
    
    # combined_logs = pd.concat([strg_comb, mem_comb], ignore_index=True)
    
    combined_logs = pd.concat([strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec], ignore_index=True)
    combined_logs.drop(['unix_time_s', 'unix_time_ns'], axis=1, inplace=True)
    
    return combined_logs

In [124]:
def generate_dataset(HWR_PATH, DATASET, operations_logs_to_load, kaggle_files_to_load, window_s: int, epsilon_s: int, ransomware):
    final_dataset = pd.DataFrame()
    for operation, operation_logs in operations_logs_to_load.items():
        print(f"Reading {operation} logs from Kaggle")
        strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec = read_kaggle_access_file(HWR_PATH, DATASET, operation_logs, kaggle_files_to_load)
        print(f"Combining {operation} log files")
        combined_log_files = generate_combined_log_file(strg_read, strg_write, mem_read, mem_write, mem_read_write, mem_exec)
        print(f"Creating features using {operation} log files")
        feature_dataset = ransmap_preprocessing(combined_log_files, operation in ransomware, operation, window_s, epsilon_s)
        print(f"Finished Creating feature dataset for {operation}\n")
        final_dataset = pd.concat([final_dataset, feature_dataset], ignore_index=True)
    
    return final_dataset

In [19]:
operations_logs_to_load

{'AESCrypt': ['AESCrypt/AESCrypt-20230524_20-18-05'],
 'Conti': ['Conti/Conti-20230426_23-09-03'],
 'Darkside': ['Darkside/Darkside-20230512_00-06-29'],
 'Firefox': ['Firefox/Firefox-20230428_00-35-50'],
 'Idle': ['Idle/Idle-20230518_22-56-17'],
 'LockBit': ['LockBit/LockBit-20230517_21-35-46'],
 'Office': ['Office/Office-20230525_23-11-40'],
 'REvil': ['REvil/REvil-20230427_23-00-37'],
 'Ryuk': ['Ryuk/Ryuk-20230510_23-53-48'],
 'SDelete': ['SDelete/SDelete-20230519_00-20-40'],
 'WannaCry': ['WannaCry/WannaCry-20230510_21-11-33'],
 'Zip': ['Zip/Zip-20230524_22-57-19']}

In [125]:
final_dataset = generate_dataset(HWR_PATH, DATASET, operations_logs_to_load, kaggle_files_to_load, 30, 1, ransomware)

Reading AESCrypt logs from Kaggle
Combining AESCrypt log files
Creating features using AESCrypt log files
Finished Creating feature dataset for AESCrypt

Reading Conti logs from Kaggle
Combining Conti log files
Creating features using Conti log files
Finished Creating feature dataset for Conti

Reading Darkside logs from Kaggle


100%|██████████| 5.26M/5.26M [00:00<00:00, 9.88MB/s]


100%|██████████| 16.4M/16.4M [00:01<00:00, 14.0MB/s]


100%|██████████| 17.3M/17.3M [00:01<00:00, 15.5MB/s]


100%|██████████| 18.9M/18.9M [00:01<00:00, 16.7MB/s]


...

100%|██████████| 6.43M/6.43M [00:00<00:00, 11.9MB/s]


100%|██████████| 19.6k/19.6k [00:00<00:00, 1.01MB/s]


100%|██████████| 7.65k/7.65k [00:00<00:00, 3.91MB/s]

Combining Zip log files
Creating features using Zip log files


Finished Creating feature dataset for Zip



In [126]:
final_dataset.head()

,start,end,avg_ata_read,avg_ata_write,lba_read_var,lba_write_var,avg_storage_entropy,avg_entropy_mem_write,avg_entropy_mem_read_write,num_4KB_pages_mem_read,...,num_MIMO_pages_mem_read,num_MIMO_pages_mem_write,num_MIMO_pages_mem_read_write,num_MIMO_pages_mem_exec,gpa_mem_read_var,gpa_mem_write_var,gpa_mem_read_write_var,gpa_mem_exec_var,is_ransomware,operation
0,0 days 00:00:00,0 days 00:00:30,2.218035e+06,156074.666667,6.906892e+14,3.467378e+15,0.339909,0.008179,0.757494,10,...,18824,19116,0,0,6.087713e+18,1.611204e+19,3.653646e+19,3.859055e+19,False,AESCrypt
1,0 days 00:00:01,0 days 00:00:31,2.369860e+06,287010.133333,6.640260e+14,2.397020e+15,0.356280,0.006177,0.625245,3,...,19660,19955,0,0,6.797337e+17,3.460737e+17,2.335112e+19,2.666074e+19,False,AESCrypt
2,0 days 00:00:02,0 days 00:00:32,2.365577e+06,302028.800000,6.595067e+14,2.377311e+15,0.361597,0.005718,0.689899,2,...,19799,20092,0,0,3.764257e+17,2.918401e+17,2.929815e+19,4.008223e+18,False,AESCrypt
3,0 days 00:00:03,0 days 00:00:33,2.365577e+06,277725.866667,6.595067e+14,2.243987e+15,0.325978,0.005725,0.689899,2,...,19641,19932,0,0,3.794003e+17,2.925154e+17,2.929815e+19,4.008223e+18,False,AESCrypt
4,0 days 00:00:04,0 days 00:00:34,2.364621e+06,277179.733333,6.597213e+14,2.245231e+15,0.325634,0.005700,0.689899,2,...,19626,19918,0,0,3.813220e+17,2.928641e+17,2.929815e+19,4.008223e+18,False,AESCrypt


In [127]:
len(final_dataset)

2148

In [130]:
final_dataset['is_ransomware'].unique()

array([False,  True])

In [133]:
final_dataset_file_path = os.path.join('processed_dataset', 'final_dataset_with_first_logs.pkl')
with open(final_dataset_file_path, 'wb') as file:
    pickle.dump(final_dataset, file)


In [132]:
final_dataset.index

RangeIndex(start=0, stop=2148, step=1)